# colab_02 — SEA-AD: download + per-study QC

Second study in the substrate. **SEA-AD** (Seattle Alzheimer's Disease Brain Cell Atlas) — 84 donors, AWS Open Data (no DUC), 1.2M+ nuclei, fully annotated. Source: public AWS bucket `sea-ad-single-cell-profiling`.

**How this differs from `colab_01` (Li 2025) — read before running.** Li shipped *raw per-sample 10x* (Cell-Ranger-filtered, no doublet removal, no annotation). SEA-AD ships a single **consortium-QC'd, doublet-removed, cell-type-annotated** `final-nuclei` object per region. That changes three things:

1. **No per-sample 10x stacking** — one h5ad per region, loaded directly. The raw-barcode-prefilter dance from `colab_01 4a` is unnecessary.
2. **Doublets are inherited, not recomputed** — re-running Scrublet on an already-doublet-removed object would double-dip and break cross-study comparability of the QC funnel. Scrublet is **gated off by default** (`RUN_SCRUBLET=False`, cell 5c); the two-pass implementation is kept for the case where a pre-doublet-removal object is substituted.
3. **Cell-type labels exist** — so the mito-sensitivity sweep (5d) runs with *real* astro/microglia labels, not totals-only. This lets the niche-level mito check (carried forward from `colab_01`'s 2026-06-05 decision) run *here* at per-study QC for SEA-AD, which Li could not do.

**Still uniform across studies:** the locked secondary-QC thresholds (min 500 counts, min 250 genes, max 5% mito, min 10 cells/gene) are re-applied on top of the consortium's QC — the "two-layer" principle (inherit published QC + labels, then a uniform secondary filter for cross-study consistency).

**Three config decisions are surfaced as named flags at the top of the relevant cells — confirm them before running:**
- **Region** (3a, `REGIONS`): MTG only vs MTG + DLPFC.
- **Disease label** (4b, `DIAGNOSIS_FROM`): how SEA-AD's neuropathology *continuum* is reduced to control / intermediate / AD. Nothing is dropped at QC — the binary contrast is deferred to the eval step.
- **Scrublet** (5c, `RUN_SCRUBLET`): off by default (consortium already removed doublets).

**Output:** `seaad_processed.h5ad` (Drive, gitignored) + Audit A (SEA-AD row) + Audit B flip (APOE recovered) + env snapshot, all committed to GitHub.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Same pattern as `colab_00`/`colab_01`: Drive mount, runtime-local repo clone, `requirements_integration.txt` install. Asserts Python 3.12 (scvi-tools 1.4.x requires it). Adds repo to `sys.path` so `from src.* import ...` resolves.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels for pandas, scipy, etc.
# Without this, pip resolves against Colab's pre-installed numpy 2.x, then
# downgrades numpy -- leaving a binary ABI mismatch that crashes on import.
# `-q` removed so dependency-conflict warnings stay visible.
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt

## 2 — Environment capture

### 2a — pip freeze + env JSON

Writes `colab_02_<date>_pip_freeze.txt` and `colab_02_<date>_env.json` under `outputs/software_versions/`. **Normalization fix vs `colab_01`:** restores the `cuda_available` and `cudnn_version` keys that `colab_00` captured but `colab_01` dropped — keeps the env schema stable across notebooks.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_02"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot = {
    "notebook_id":      NOTEBOOK_ID,
    "date":             TODAY,
    "python_version":   sys.version,
    "platform":         platform.platform(),
    "os_release":       platform.release(),
    "gpu":              _run(["nvidia-smi", "-L"]),
    "nvidia_driver":    _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":       _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
    env_snapshot["cuda_available"]     = bool(torch.cuda.is_available())
    env_snapshot["cudnn_version"]      = torch.backends.cudnn.version() if torch.cuda.is_available() else None
except ImportError:
    env_snapshot["torch_version"]  = None
    env_snapshot["cuda_available"] = None
    env_snapshot["cudnn_version"]  = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Download SEA-AD

SEA-AD single-cell data live in a **public AWS Open Data bucket** (`sea-ad-single-cell-profiling`) — anonymous access, no DUC, no registration. The bucket is listed with `--no-sign-request`; **the listing is the source of truth** for object keys (the consortium re-dates files on re-release, so a hard-coded filename rots). We resolve the newest `*final-nuclei*.h5ad` per region at runtime and download it to runtime-local disk (`/content`, not Drive — keeps the ~6 GB/region raw object off the Drive budget; deleted in 7b).

### 3a — Resolve + download per-region `final-nuclei` h5ad

**DECISION A — `REGIONS`.** SEA-AD released two regions: **MTG** (Middle Temporal Gyrus — the flagship 1.24M-nuclei atlas) and **DLPFC** (prefrontal, bucket prefix `PFC/`). Default below is **MTG only** — a clean first SEA-AD pass mirroring `colab_01`'s "smallest substrate first" smoke-test logic, and it avoids introducing region as an integration covariate before the pipeline is proven. Set `REGIONS = ["MTG", "DLPFC"]` to process both (the locked substrate wants cross-region; this is the fast-follow once MTG QC is clean).

**Cohort vs Reference (confirmed from the live listing).** Each region prefix holds *two* `final-nuclei` objects: **`SEAAD_*`** (the 84-donor AD continuum — its low-ADNC donors are our controls, its high-ADNC donors our AD; ~36 GB for MTG) and **`Reference_*`** (the neurotypical Allen mapping reference — younger brains, *not* our control group; ~6 GB). The selector matches `seaad` explicitly and never falls back to `Reference_`. **Note the size:** the MTG cohort object is ~36 GB / ~1.24M cells (the earlier ~6 GB storage estimate had mistaken the Reference file). The download skips if the runtime already holds the file; 4a reads it in **backed mode** + downsamples, so the 36 GB never fully materializes.

In [ ]:
import os, sys, subprocess, glob

# --- DECISION A: which SEA-AD region(s) -----------------------------------------
REGIONS = ["MTG"]            # or ["MTG", "DLPFC"]

SEAAD_BUCKET  = "sea-ad-single-cell-profiling"
REGION_PREFIX = {            # region -> S3 prefix under the bucket
    "MTG":   "MTG/RNAseq/",
    "DLPFC": "PFC/RNAseq/",  # SEA-AD labels the prefrontal release "PFC"
}
RAW_DIR = "/content/seaad_raw"
os.makedirs(RAW_DIR, exist_ok=True)

# awscli ships on Colab, but install defensively (anonymous S3 needs it).
if subprocess.run(["which", "aws"], capture_output=True).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "awscli"], check=True)

def _s3_ls(prefix):
    r = subprocess.run(
        ["aws", "s3", "ls", "--no-sign-request", f"s3://{SEAAD_BUCKET}/{prefix}"],
        capture_output=True, text=True)
    return r.stdout

region_files = {}
for region in REGIONS:
    prefix = REGION_PREFIX[region]
    listing = _s3_ls(prefix)
    print(f"=== s3://{SEAAD_BUCKET}/{prefix} ===")
    print(listing or "(empty -- prefix wrong? inspect bucket layout below)")
    # object keys are the last whitespace-delimited token on each line
    keys = [ln.split()[-1] for ln in listing.splitlines() if ln.strip().endswith(".h5ad")]
    # Pick the SEA-AD COHORT final-nuclei object. The bucket holds TWO final-nuclei
    # files: 'SEAAD_*' (the 84-donor AD continuum -- our controls+AD) and
    # 'Reference_*' (the neurotypical Allen mapping reference -- younger brains, NOT
    # our controls). Match 'seaad' explicitly; never fall back to Reference_.
    final  = [k for k in keys if "final-nuclei" in k.lower()]
    cohort = [k for k in final if "seaad" in os.path.basename(k).lower()]
    chosen = sorted(cohort or [k for k in final if "reference" not in os.path.basename(k).lower()])
    if not chosen:
        raise RuntimeError(
            f"No SEA-AD cohort final-nuclei .h5ad under {prefix} (found final={final!r}). "
            f"Inspect the listing above and adjust the filter / REGION_PREFIX[{region!r}].")
    key = chosen[-1]                       # newest by lexical date suffix
    dest = os.path.join(RAW_DIR, f"{region}__{os.path.basename(key)}")
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        # ~36 GB/region -- never re-download if the runtime already has it.
        print(f"\n--> {region}: already present, skipping download "
              f"({os.path.getsize(dest)/1024**3:.2f} GB) -- {os.path.basename(dest)}")
    else:
        print(f"\n--> {region}: downloading {key}")
        subprocess.run(
            ["aws", "s3", "cp", "--no-sign-request",
             f"s3://{SEAAD_BUCKET}/{prefix}{key}", dest], check=True)
    region_files[region] = dest

print("\nDownloaded:")
for r, p in region_files.items():
    print(f"  {r}: {p}  ({os.path.getsize(p)/1024**3:.2f} GB)")

## 4 — Load SEA-AD into AnnData + harmonize metadata

### 4a — Load region h5ad(s), tag region, promote RAW COUNTS into `.X`

**Scale strategy — backed read + downsample (DECIDED 2026-06-05).** The MTG cohort object is ~1.24M cells / ~36 GB; a full `sc.read_h5ad` into RAM OOMs Colab, and at 1.24M SEA-AD would dominate the 3-study integration (Li ~382k, Haney ~100k → trips the ">60% of held-out" balance audit). So the cell opens the file **backed** (`backed="r"` — obs/var in RAM, X on disk), builds a stratified index from obs that **keeps every astrocyte + microglia** (the rare niche substrate) and **caps the abundant non-glia at `NONGLIA_PER_DONOR` per donor**, then materializes *only* that subset. This lands SEA-AD ≈ Li's scale, glia-complete, and the 36 GB X is never fully in RAM. `NONGLIA_PER_DONOR` is the RAM knob (lower if `to_memory` is tight, raise on an 83 GB A100 box). `DOWNSAMPLE=False` attempts the full object (needs a backed-stream rework — not wired). All cell types are kept (just thinned) so ITS integration still sees the full taxonomy. Downsample provenance is recorded to the QC trace in 5b.

**Critical correctness step (raw counts).** SEA-AD / CELLxGENE-convention objects store *normalized* values in `.X` and **raw integer counts** in a layer (`UMIs`/`counts`) or in `.raw`. Scrublet (if run) and scVI both require raw counts, so after materializing the subset this cell promotes raw counts into `.X` and **fails loud** if it cannot find a counts matrix. The object is then slimmed (drop SEA-AD's normalized layers / own embeddings / `.raw` / graphs / `uns` — all recomputed downstream) so the saved processed h5ad stays small. Multi-region runs concat with `join="outer"` and tag `region` on obs.

The `_looks_like_counts` guard is deliberately strict — a naive "are the values integers?" check is not enough, because (a) SEA-AD objects are **sorted by cell type**, so a head sample is an atypical subpopulation (it samples a spread of rows instead), and (b) library-size-normalized data can round to integers yet still be wrong. The decisive test is that **per-cell totals vary** — `normalize_total` forces every cell to the same total, so equal row sums are the signature of normalized data even when it looks integer-valued. The `.raw` promotion path aligns genes by name on the intersection and **asserts** exact alignment (var-names aren't deduped until after concat, so a silent gene/count misalignment is otherwise possible). A counts-sanity line (`nnz_max`, `median_cell_total`, `cell_total_CV`) prints per region so the matrix can be eyeballed.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from scipy.sparse import issparse, vstack
import h5py

# anndata lazy-IO helpers moved across versions -- resolve robustly.
try:
    from anndata import read_elem
except Exception:
    from anndata.experimental import read_elem
try:
    from anndata.experimental import sparse_dataset
except Exception:
    from anndata.io import sparse_dataset

# --- STREAMING DOWNSAMPLE (RAM-safe) --------------------------------------------
# SEA-AD MTG final-nuclei is ~1.24M cells / ~36 GB and ALSO ships a full .raw
# (raw counts) plus normalized layers. sc.read_h5ad(..., backed="r") only backs
# .X -- it EAGERLY loads .raw + .layers + .obsm into RAM at read time, OOMing
# Colab before any subsetting. So we bypass the full AnnData entirely: read
# obs/var with read_elem, wrap ONLY the counts matrix as a lazy sparse_dataset,
# and pull just the rows we keep (all astro+micro glia + a per-donor cap of the
# abundant non-glia). Peak RAM ~= the kept subset (~400k rows), never 1.24M.
# This is the streaming rework the backed-mode comment used to defer.
DOWNSAMPLE        = True
NONGLIA_PER_DONOR = 2000
DOWNSAMPLE_SEED   = 0
READ_BATCH        = 50000          # rows per backed read -- RAM knob (lower if tight)

def _looks_like_counts(X, n_rows=500):
    """Is X raw integer counts (not normalized)? Works on an in-memory matrix OR a
    backed sparse_dataset (both support fancy row indexing + .sum). SEA-AD is
    SORTED by cell type, so sample a SPREAD of rows. Requires together:
    non-negative, integral, has values > 1, and per-cell totals NOT all equal --
    sc.pp.normalize_total makes every row sum to the same target, so equal row
    totals == normalized even if values rounded to integers (the decisive test)."""
    n = X.shape[0]
    if n == 0:
        return False
    idx = np.unique(np.linspace(0, n - 1, min(n_rows, n)).astype(int))
    sub = X[idx]
    data = (sub.data if issparse(sub) else np.asarray(sub).ravel())
    data = data[data != 0]
    if data.size == 0:
        return False
    nonneg   = bool(np.all(data >= 0))
    integral = bool(np.allclose(data, np.round(data)))
    has_gt1  = bool(data.max() > 1.0)
    rowsums  = np.asarray(sub.sum(axis=1)).ravel()
    rowsums  = rowsums[rowsums > 0]
    varied   = rowsums.size > 0 and float(np.std(rowsums) / np.mean(rowsums)) > 0.01
    return nonneg and integral and has_gt1 and varied

def _resolve(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

def _locate_counts(f):
    """Return (xkey, vkey, backed_X) for whichever matrix holds raw integer counts.
    Probe order: /X, then each /layers/*, then /raw/X. /X and layers share the main
    /var gene axis; .raw carries its own /raw/var -- hence the paired var key. Only
    CSR is row-streamable here; CSC/dense are reported and skipped. Fail loud with a
    full structure + per-candidate verdict dump so a miss is self-diagnosing."""
    cands = [("X", "var")]
    if "layers" in f:
        cands += [(f"layers/{k}", "var") for k in f["layers"].keys()]
    if "raw" in f and "X" in f["raw"]:
        cands.append(("raw/X", "raw/var"))
    report = []
    for xkey, vkey in cands:
        node = f[xkey]
        enc = (node.attrs.get("encoding-type", "?")
               if isinstance(node, h5py.Group) else "dense")
        if not str(enc).startswith("csr"):
            report.append(f"{xkey}: enc={enc} (skipped -- not CSR-streamable)")
            continue
        Xds = sparse_dataset(node)
        ok = _looks_like_counts(Xds)
        report.append(f"{xkey}: csr shape={Xds.shape} looks_like_counts={ok}")
        if ok:
            return xkey, vkey, Xds
    raise RuntimeError(
        "No raw integer counts found. scVI + Scrublet need raw counts, not "
        "normalized values.\nProbed:\n  " + "\n  ".join(report) +
        f"\n  top-level={list(f.keys())}"
        f"\n  layers={list(f['layers'].keys()) if 'layers' in f else None}"
        f"\n  raw={list(f['raw'].keys()) if 'raw' in f else None}")

def _read_rows_batched(Xds, rows):
    """Materialize only `rows` from a backed CSR, in batches to bound peak RAM."""
    rows = np.asarray(rows)
    chunks = [Xds[rows[i:i + READ_BATCH]] for i in range(0, len(rows), READ_BATCH)]
    return (vstack(chunks).tocsr() if len(chunks) > 1 else chunks[0].tocsr())

parts = []
ds_n_full = ds_n_kept = ds_glia = ds_excl_uia = ds_excl_ref = 0
for region, path in region_files.items():
    with h5py.File(path, "r") as f:
        obs = read_elem(f["obs"])                       # full obs (cat codes -> light)
        n_full = obs.shape[0]
        xkey, vkey, Xds = _locate_counts(f)
        var = read_elem(f[vkey])
        print(f"{region}: counts at /{xkey}  (genes from /{vkey}, n_var={var.shape[0]})")

        # --- Consortium inclusion gate (applies to ALL branches) -----------------
        # SEA-AD's final-nuclei object still carries cells the consortium did NOT
        # use in analysis (e.g. amplification-fail) and 5 neurotypical REFERENCE
        # donors (a different sampling frame, no APOE genotype). Exclude both BEFORE
        # selection so "glia-complete" means analysis-grade glia only. Fail loud if
        # either flag column is absent (SEA-AD schema drift must not pass silently).
        uia_col = _resolve(obs.columns, "Used in analysis")
        ref_col = _resolve(obs.columns, "Neurotypical reference")
        assert uia_col and ref_col, (
            f"expected 'Used in analysis' + 'Neurotypical reference' in obs; got "
            f"uia={uia_col!r} ref={ref_col!r}; have {list(obs.columns)}")
        def _istrue(col):
            return obs[col].astype(str).str.strip().str.lower().isin(
                ["true", "yes", "y", "1", "t"]).to_numpy()
        used_mask = _istrue(uia_col)
        ref_mask  = _istrue(ref_col)
        eligible  = used_mask & ~ref_mask
        n_drop_uia = int((~used_mask).sum())
        n_drop_ref = int((used_mask & ref_mask).sum())   # reference among otherwise-used
        ds_excl_uia += n_drop_uia
        ds_excl_ref += n_drop_ref
        print(f"{region} inclusion: drop {n_drop_uia} not-used-in-analysis ({uia_col!r}) "
              f"+ {n_drop_ref} neurotypical-reference ({ref_col!r}) -> "
              f"{int(eligible.sum())} / {n_full} eligible")
        pos = np.arange(n_full)

        if DOWNSAMPLE:
            ct_col = _resolve(obs.columns, "Subclass", "subclass", "subclass_label",
                              "Supertype", "supertype", "Class")
            dn_col = _resolve(obs.columns, "Donor ID", "donor_id",
                              "external_donor_name", "individualID")
            assert ct_col and dn_col, (
                f"downsample needs cell-type + donor cols; got ct={ct_col!r} "
                f"dn={dn_col!r}; have {list(obs.columns)}")
            ct = obs[ct_col].astype(str).str.lower()
            glia_raw     = (ct.str.contains("astro") | ct.str.contains("micro")).to_numpy()
            glia_mask    = glia_raw & eligible            # keep ALL eligible glia
            nonglia_mask = ~glia_raw & eligible           # cap eligible non-glia
            glia_pos = pos[glia_mask]
            ng = pd.DataFrame({"pos": pos[nonglia_mask],
                               "donor": obs[dn_col].to_numpy()[nonglia_mask]})
            samp = ng.groupby("donor", group_keys=False)["pos"].apply(
                lambda s: s.sample(n=min(len(s), NONGLIA_PER_DONOR),
                                   random_state=DOWNSAMPLE_SEED))
            keep_pos = np.sort(np.concatenate([glia_pos, samp.to_numpy()]))
            print(f"{region} downsample: all-glia={len(glia_pos)} + non-glia "
                  f"(cap {NONGLIA_PER_DONOR}/donor)={len(samp)} -> {len(keep_pos)} / "
                  f"{n_full}  (cell-type={ct_col!r}, donor={dn_col!r})")
            ds_glia   += int(len(glia_pos))
            ds_n_kept += int(len(keep_pos))
        else:
            keep_pos = pos[eligible]                      # WARNING: full read -> may OOM
            ds_n_kept += int(eligible.sum())
        ds_n_full += n_full

        X = _read_rows_batched(Xds, keep_pos)            # ONLY kept rows enter RAM
        a = ad.AnnData(X=X, obs=obs.iloc[keep_pos].copy(), var=var.copy())
        del obs, X

    a.obs["region"] = region
    print(f"{region}: {a.shape}  (streamed subset, raw counts)")
    # Counts sanity line -- integer-valued nonzeros at counts scale, per-cell
    # totals that VARY (not all equal). Confirms we grabbed raw, not normalized.
    if issparse(a.X):
        _d  = a.X.data
        _rs = np.asarray(a.X.sum(axis=1)).ravel()
        print(f"   X check: nnz_min={_d.min():.3g} nnz_max={_d.max():.3g} "
              f"median_cell_total={np.median(_rs):.0f} "
              f"cell_total_CV={np.std(_rs)/np.mean(_rs):.2f}")
    parts.append(a)

DOWNSAMPLE_INFO = {
    "applied":               bool(DOWNSAMPLE),
    "n_full":                int(ds_n_full),
    "n_kept":                int(ds_n_kept),
    "glia_all_kept":         int(ds_glia),
    "excluded_not_used_in_analysis":   int(ds_excl_uia),
    "excluded_neurotypical_reference": int(ds_excl_ref),
    "nonglia_cap_per_donor": NONGLIA_PER_DONOR if DOWNSAMPLE else None,
    "seed":                  DOWNSAMPLE_SEED,
    "scheme":                "stream-read; keep all astro+micro; cap non-glia per donor",
}

if len(parts) == 1:
    adata = parts[0]
else:
    adata = ad.concat(parts, join="outer", label="region_concat", index_unique="-")
del parts

adata.var_names_make_unique()
print(f"\nLoaded: {adata}")
print(f"Downsample: {DOWNSAMPLE_INFO}")
fmt = adata.X.getformat() if issparse(adata.X) else type(adata.X).__name__
print(f"X dtype: {adata.X.dtype}  format: {fmt}")
try:
    import psutil
    _m = psutil.virtual_memory()
    print(f"[RAM] after 4a load: {_m.used/1e9:.1f} / {_m.total/1e9:.1f} GB ({_m.percent:.0f}%)")
except Exception:
    pass


### 4b — Harmonize donor metadata (APOE / diagnosis / sex) into the project schema

SEA-AD ships rich donor metadata **already on `adata.obs`** (unlike Li, which needed a series-matrix parse). This cell maps SEA-AD's column names onto the project's canonical schema — `donor_id`, `apoe_genotype`, `apoe_carrier`, `diagnosis`, `sample_id` — using defensive name resolution (SEA-AD column casing varies by release).

**APOE carrier parsing.** `_carrier` matches allele **digits** (`re.findall(r"[234]")`), not a literal `"E4"` substring — SEA-AD's "APOE Genotype" column is often bare-numeric (`"3/4"`, `"4/4"`), on which a substring check would silently miss every carrier and collapse the project's primary axis. Missing/unparseable genotypes map to `"unknown"` (warned, excluded from carrier counts), never silently folded into the `"noncarrier"` baseline.

**DECISION B — `DIAGNOSIS_FROM`.** SEA-AD is a *continuum* cohort, not binary case/control. We derive a 3-level `diagnosis` ∈ {control, intermediate, AD} and **keep all donors** — nothing is dropped at QC. The binary control-vs-late-AD contrast (locked: Braak V–VI = AD) is deferred to the eval/integration step, which decides whether to drop `intermediate` or treat the axis ordinally. Default derives from **ADNC** (Overall AD Neuropathological Change: Not AD/Low → control, Intermediate → intermediate, High → AD); set `DIAGNOSIS_FROM="braak"` to derive from Braak stage (0–II → control, III–IV → intermediate, V–VI → AD) instead. The cell prints a **raw-category → derived-label crosstab** so the continuum collapse (e.g. ADNC "Low" folding into control) is auditable rather than buried in a string match. The CELLxGENE `"disease"` column is deliberately *not* a fallback for ADNC — it carries a different vocabulary and would map everything to `"unknown"`.

In [ ]:
import pandas as pd
import re

DIAGNOSIS_FROM = "adnc"          # "adnc" or "braak"

def _col(df, *cands, required=True):
    low = {c.lower(): c for c in df.columns}
    for cand in cands:
        if cand.lower() in low:
            return low[cand.lower()]
    if required:
        raise KeyError(f"obs missing any of {cands}; have {list(df.columns)}")
    return None

obs = adata.obs

donor_col = _col(obs, "Donor ID", "donor_id", "external_donor_name", "individualID")
apoe_col  = _col(obs, "APOE Genotype", "apoe_genotype", "APOE", "Apoe")
sex_col   = _col(obs, "Sex", "sex", "Gender")

adata.obs["donor_id"]      = obs[donor_col].astype(str)
adata.obs["apoe_genotype"] = obs[apoe_col].astype(str)
adata.obs["sex"]           = obs[sex_col].astype(str)
# one library per donor per region in SEA-AD -> sample_id keys Scrublet/batch
adata.obs["sample_id"]     = adata.obs["donor_id"] + "__" + adata.obs["region"].astype(str)

def _carrier(g):
    # Match the allele DIGITS, not a literal "E4" substring. SEA-AD's "APOE
    # Genotype" column is often bare-numeric ("3/4", "4/4"), on which `"E4" in g`
    # is silently False -> every E4 carrier mislabeled "noncarrier", which would
    # collapse the project's PRIMARY axis. re.findall handles "E3/E4", "3/4",
    # "e4/e4" alike. Missing/unparseable -> "unknown" (NOT folded into baseline).
    alleles = re.findall(r"[234]", str(g).upper())
    if not alleles:     return "unknown"
    if "4" in alleles:  return "carrier"     # any E4: E3/E4, E4/E4, E2/E4
    if "2" in alleles:  return "e2"          # E2/E3 -- protective; excluded from eval #2
    return "noncarrier"                       # E3/E3 -- baseline
adata.obs["apoe_carrier"] = adata.obs["apoe_genotype"].map(_carrier)

# --- DECISION B: diagnosis from the neuropath continuum -------------------------
def _from_adnc(v):
    s = str(v).strip().lower()
    if "not ad" in s or s in {"reference", "control"} or s.startswith("low"):
        return "control"
    if s.startswith("intermediate"): return "intermediate"
    if s.startswith("high"):         return "AD"
    return "unknown"

def _braak_num(v):
    # Ordered longest-first so 'III' is matched before 'II'/'I'. Correct for clean
    # single-stage values ("Braak V", "Braak III"); a transitional range like
    # "Braak III-IV" resolves to the UPPER stage (first match wins). DORMANT under
    # DIAGNOSIS_FROM="adnc" -- revisit this resolution rule before flipping to braak.
    s = str(v).upper()
    roman = {"VI":6,"IV":4,"III":3,"II":2,"V":5,"I":1,"0":0}
    for r in ["VI","IV","III","II","V","I","0"]:
        if r in s: return roman[r]
    m = re.search(r"\d", s)
    return int(m.group()) if m else None

def _from_braak(v):
    n = _braak_num(v)
    if n is None:        return "unknown"
    if n <= 2:           return "control"
    if n <= 4:           return "intermediate"
    return "AD"

if DIAGNOSIS_FROM == "adnc":
    # Do NOT add CELLxGENE's "disease" column as a fallback: it carries a
    # different vocabulary ("Alzheimer disease"/"normal"/"dementia"), not ADNC
    # levels, and would silently map every donor to "unknown". Fail loud instead.
    adnc_col = _col(obs, "Overall AD neuropathological Change",
                    "ADNC", "Overall AD neuropathologic Change")
    adata.obs["diagnosis"] = obs[adnc_col].map(_from_adnc).values
    DX_SRC = adnc_col
else:
    braak_col = _col(obs, "Braak", "Braak stage", "Braak Stage", "braak")
    adata.obs["diagnosis"] = obs[braak_col].map(_from_braak).values
    DX_SRC = braak_col

# Fail-loud donor-level sanity cross-tabs.
donor_tab = adata.obs[["donor_id","diagnosis","apoe_genotype","apoe_carrier",DX_SRC]].drop_duplicates("donor_id")
n_donors = donor_tab["donor_id"].nunique()
print(f"donors: {n_donors}   diagnosis source: {DX_SRC!r}")
if "unknown" in set(adata.obs["diagnosis"]):
    n_unk = donor_tab[donor_tab["diagnosis"]=="unknown"]["donor_id"].nunique()
    print(f"WARNING: {n_unk} donor(s) -> diagnosis='unknown' (unmapped category in {DX_SRC!r}); "
          "inspect the cross-tab and extend the mapper.")
if "unknown" in set(adata.obs["apoe_carrier"]):
    n_unk_a = donor_tab[donor_tab["apoe_carrier"]=="unknown"]["donor_id"].nunique()
    print(f"WARNING: {n_unk_a} donor(s) -> apoe_carrier='unknown' (missing/unparseable genotype); "
          "excluded from carrier counts, not folded into noncarrier.")
# Make the continuum -> {control/intermediate/AD} mapping AUDITABLE: which raw
# neuropath categories folded into which label (e.g. ADNC 'Low' -> control).
print(f"\n=== donor counts: raw {DX_SRC!r} -> derived diagnosis ===")
print(pd.crosstab(donor_tab[DX_SRC], donor_tab["diagnosis"]))
print("\n=== donor counts: diagnosis x apoe_carrier (E2 kept separate) ===")
print(pd.crosstab(donor_tab["diagnosis"], donor_tab["apoe_carrier"]))
print("\nobs schema columns now:",
      [c for c in ["donor_id","sample_id","region","apoe_genotype","apoe_carrier","diagnosis","sex"]])

## 5 — Secondary QC (locked defaults)

Uniform secondary filter re-applied on top of SEA-AD's consortium QC (the two-layer principle): min 500 counts/nucleus, min 250 genes/nucleus, max 5% mito, min 10 cells/gene. Scrublet inherited (5c). Because the object is already QC'd, expect the count/gene/mito steps to drop *few* additional cells — the funnel here is mostly a cross-study-consistency formality, and a divergent drop would itself be a red flag worth reading.

### 5a — Annotate mito % and QC metrics

Detects var-name format (HGNC `MT-*` vs Ensembl `ENSG...`). SEA-AD objects are typically Ensembl-indexed with a `feature_name`/`gene_name` symbol column — set via `SYMBOL_COL` candidates below.

In [ ]:
looks_ensembl = adata.var_names.str.startswith("ENSG").mean() > 0.5

SYMBOL_COL = None
if looks_ensembl:
    for cand in ["feature_name", "gene_name", "gene_symbols", "gene_symbol", "Symbol"]:
        if cand in adata.var.columns:
            SYMBOL_COL = cand
            break
    if SYMBOL_COL is None:
        raise RuntimeError(
            "var_names are Ensembl but no symbol column found among "
            "[feature_name, gene_name, gene_symbols, ...]. Inspect adata.var.columns.")
    adata.var["mt"] = adata.var[SYMBOL_COL].astype(str).str.upper().str.startswith("MT-")
else:
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
if int(adata.var["mt"].sum()) == 0:
    raise RuntimeError(
        "Zero mito genes detected. If SYMBOL_COL is set, verify it holds HGNC names "
        "(MT- prefix), not Ensembl IDs.")

print(f"shape:      {adata.shape}")
print(f"var format: {'Ensembl (symbols in ' + str(SYMBOL_COL) + ')' if looks_ensembl else 'HGNC symbols'}")
print(f"mito genes: {int(adata.var['mt'].sum())}")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())

### 5b — Apply locked thresholds (single-copy mask)

Same memory-disciplined pattern as `colab_01 5b`: the three per-cell thresholds are boolean masks on `obs` (5a already computed the metrics), combined into ONE mask and ONE matrix copy. Snapshots `pre_mito_obs` (post count/gene filter, pre mito filter) for 5d's sweep. `N_RAW_BARCODES_TOTAL` is `None` here — SEA-AD has no per-sample prefilter denominator (it's a single pre-filtered object), so the initial cell count is the honest start.

In [ ]:
import gc, json
try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:20s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except Exception:
    def _ram(tag):
        print(f"[RAM] {tag}: psutil unavailable")

QC_PARAMS = {
    "min_counts":         500,
    "min_genes":          250,
    "max_mito_pct":       5.0,
    "min_cells_per_gene": 10,
}
N_RAW_BARCODES_TOTAL = None          # SEA-AD ships one pre-filtered object (no per-sample prefilter)
n_cells_0 = adata.n_obs
n_genes_0 = adata.n_vars

_ram("5b start")
counts_mask = (adata.obs["total_counts"]      >= QC_PARAMS["min_counts"]).to_numpy()
genes_mask  = (adata.obs["n_genes_by_counts"] >= QC_PARAMS["min_genes"]).to_numpy()
mito_mask   = (adata.obs["pct_counts_mt"]     <  QC_PARAMS["max_mito_pct"]).to_numpy()

n_after_counts = int(counts_mask.sum())
n_after_genes  = int((counts_mask & genes_mask).sum())
cell_keep      = counts_mask & genes_mask & mito_mask
n_after_mito   = int(cell_keep.sum())

pre_mito_obs = adata.obs[counts_mask & genes_mask].copy()    # 5d sweep pool (obs only)

adata = adata[cell_keep].copy()
gc.collect()
_ram("after cell filter")

sc.pp.filter_genes(adata, min_cells=QC_PARAMS["min_cells_per_gene"])
gc.collect()
_ram("after gene filter")

qc_trace = {
    "study":                    "SEA-AD",
    "regions":                  REGIONS,
    "params":                   QC_PARAMS,
    "downsample":               DOWNSAMPLE_INFO,
    "n_cells_initial":          n_cells_0,
    "n_cells_initial_note":     "post-downsample (see downsample.n_full for pre-downsample total)",
    "prefiltered_in_4a":        False,
    "n_cells_after_min_counts": n_after_counts,
    "n_cells_after_min_genes":  n_after_genes,
    "n_cells_after_max_mito":   n_after_mito,
    "n_cells_final":            adata.n_obs,
    "n_genes_initial":          n_genes_0,
    "n_genes_final":            adata.n_vars,
}
print(json.dumps(qc_trace, indent=2))

### 5c — Doublets: inherit consortium removal (Scrublet gated off)

SEA-AD's `final-nuclei` object is **already doublet-removed** by the consortium pipeline. Re-running Scrublet and filtering would (a) double-dip and (b) make SEA-AD's QC funnel non-comparable to Li's (where Scrublet *was* the doublet step on raw 10x). So the default path **inherits** their removal and only *reports* any SEA-AD-native doublet/QC annotation it can find.

`RUN_SCRUBLET=True` re-enables the full two-pass Scrublet (identical logic to `colab_01 5c`: compute per sample → rescue auto-threshold failures at cohort-median rate → single filter) — use it only if a *pre*-doublet-removal SEA-AD object is ever substituted.

In [ ]:
import numpy as np
import pandas as pd

RUN_SCRUBLET = False     # SEA-AD final-nuclei is consortium-doublet-removed (see markdown)

# Report any SEA-AD-native doublet / QC-pass annotation (provenance, not action).
native_doublet_cols = [c for c in adata.obs.columns
                       if any(k in c.lower() for k in ["doublet", "qc", "outlier", "pass"])]
print("SEA-AD-native QC/doublet-related obs columns:", native_doublet_cols or "(none found)")
for c in native_doublet_cols:
    try:
        col = adata.obs[c]
        if pd.api.types.is_numeric_dtype(col):
            # continuous (e.g. 'Doublet score') -> quantile summary + coarse bins,
            # NOT a per-value dump (that floods the log with ~1k unique floats).
            desc = col.describe()[["min", "25%", "50%", "75%", "max"]].round(3).to_dict()
            bins = pd.cut(col, [0, 0.05, 0.1, 0.2, 0.3, 1.0],
                          include_lowest=True).value_counts().sort_index()
            print(f"  {c} (numeric): {desc}")
            print("    bins:", {str(k): int(v) for k, v in bins.items()})
        else:
            print(f"  {c}: {col.value_counts(dropna=False).to_dict()}")
    except Exception as e:
        print(f"  {c}: (summary failed: {e})")

if not RUN_SCRUBLET:
    qc_trace["doublets"] = {
        "method":        "inherited (SEA-AD consortium pipeline)",
        "scrublet_run":  False,
        "native_cols":   native_doublet_cols,
        "note":          "consortium final-nuclei object is already doublet-removed; "
                         "Scrublet skipped to avoid double-dipping and to keep the QC "
                         "funnel comparable across studies.",
    }
    print("\nScrublet skipped -- doublets inherited from SEA-AD consortium QC.")
else:
    import scrublet as scr
    from scipy.sparse import csr_matrix, issparse
    _ram("5c start")
    if not (issparse(adata.X) and adata.X.getformat() == "csr"):
        adata.X = csr_matrix(adata.X)
    if adata.X.dtype != np.float32:
        adata.X = adata.X.astype(np.float32); gc.collect()

    SAMPLE_COL = "sample_id"
    EXPECTED_DOUBLET_RATE, AUTO_FAIL_RATE = 0.10, 0.02
    adata.obs["doublet_score"]      = np.nan
    adata.obs["scrublet_auto_call"] = False
    sample_auto_rate = {}
    for sample in adata.obs[SAMPLE_COL].unique():
        mask  = (adata.obs[SAMPLE_COL] == sample).values
        sub_X = adata.X[mask]
        if sub_X.shape[0] < 30:
            print(f"  {sample}: n={sub_X.shape[0]} too small, skipping"); del sub_X; continue
        sclt = scr.Scrublet(sub_X, expected_doublet_rate=EXPECTED_DOUBLET_RATE)
        scores, calls = sclt.scrub_doublets(verbose=False)
        if calls is None:
            calls = np.zeros(sub_X.shape[0], dtype=bool)
        sample_auto_rate[sample] = float(np.mean(calls))
        adata.obs.loc[mask, "doublet_score"]      = scores
        adata.obs.loc[mask, "scrublet_auto_call"] = calls
        del sclt, scores, calls, sub_X, mask; gc.collect()

    failed = sorted(s for s, r in sample_auto_rate.items() if r <  AUTO_FAIL_RATE)
    ok     = sorted(s for s, r in sample_auto_rate.items() if r >= AUTO_FAIL_RATE)
    cohort_median_rate = float(np.median([sample_auto_rate[s] for s in ok])) if ok else EXPECTED_DOUBLET_RATE
    print(f"auto-OK: {len(ok)} | FAILED(<{AUTO_FAIL_RATE:.0%}): {len(failed)} | cohort median {cohort_median_rate:.1%}")

    adata.obs["predicted_doublet"] = adata.obs["scrublet_auto_call"].astype(bool)
    for sample in failed:
        mask = (adata.obs[SAMPLE_COL] == sample).values
        s = adata.obs.loc[mask, "doublet_score"].to_numpy()
        if np.all(np.isnan(s)):
            continue
        k   = max(int(np.ceil(cohort_median_rate * len(s))), 0)
        thr = np.sort(s)[::-1][k - 1] if k > 0 else np.inf
        adata.obs.loc[mask, "predicted_doublet"] = s >= thr

    n_before = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"].to_numpy()].copy(); gc.collect()
    qc_trace["doublets"] = {
        "method": "scrublet two-pass (RUN_SCRUBLET=True)",
        "scrublet_run": True, "removed": int(n_before - adata.n_obs),
        "failed_samples": failed, "cohort_med_rate": round(cohort_median_rate, 4),
    }
    qc_trace["n_cells_after_scrublet"] = int(adata.n_obs)
    print(f"After Scrublet: {adata.n_obs} cells ({n_before - adata.n_obs} removed)")

### 5d — Mito threshold sensitivity (3% / 5% / 10%) — with real cell-type labels

Sanity check on the locked 5% default, run on `pre_mito_obs` (post count/gene filter, pre mito filter — the pool that still contains the high-mito cells the 5% cut removes). **Unlike Li**, SEA-AD ships cell-type labels (`Subclass`/`Supertype`), so this also executes the **niche-level** check carried forward from the 2026-06-05 decision: among astrocytes and microglia specifically, does 5% preferentially cut them vs. 3%/10%? A pile-up of astro/micro loss at 5% would mean the threshold is load-bearing for the niche and needs justification (or a per-niche relaxation) before integration.

In [ ]:
MITO_THRESHOLDS = [3.0, 5.0, 10.0]

candidate_celltype_cols = [
    "Subclass", "subclass", "subclass_label", "Supertype", "supertype",
    "Class", "cell_type", "celltype", "CellType", "major_celltype", "annotation",
]
CELLTYPE_COL = next((c for c in candidate_celltype_cols if c in pre_mito_obs.columns), None)

if CELLTYPE_COL:
    _ct = pre_mito_obs[CELLTYPE_COL].astype(str).str.lower()
    astro_mask = _ct.str.contains("astro")
    micro_mask = _ct.str.contains("micro")
    print(f"Cell-type column: {CELLTYPE_COL!r}")
    print(f"Unique values: {sorted(pre_mito_obs[CELLTYPE_COL].astype(str).unique())[:40]}")
    print(f"Matched  astro={int(astro_mask.sum())}  micro={int(micro_mask.sum())}")
    if astro_mask.sum() == 0 or micro_mask.sum() == 0:
        print("WARNING: astro/micro matched 0 cells -- check the labels above (e.g. 'AST'/'MG').")
else:
    astro_mask = micro_mask = None
    print("No cell-type column found -- sweep reports total cells only.")
    print(f"Columns available: {list(pre_mito_obs.columns)}")

n_pre = len(pre_mito_obs)
sweep = {}
for t in MITO_THRESHOLDS:
    keep = pre_mito_obs["pct_counts_mt"] < t
    row = {"total_retained": int(keep.sum()), "total_frac": round(float(keep.mean()), 4)}
    if CELLTYPE_COL:
        row["astro_retained"] = int((keep & astro_mask).sum())
        row["micro_retained"] = int((keep & micro_mask).sum())
        if astro_mask.sum():
            row["astro_frac"] = round(float((keep & astro_mask).sum() / astro_mask.sum()), 4)
        if micro_mask.sum():
            row["micro_frac"] = round(float((keep & micro_mask).sum() / micro_mask.sum()), 4)
    sweep[f"mito_lt_{t}"] = row

qc_trace["mito_sensitivity"] = {
    "status":           "computed",
    "pre_mito_pool":    n_pre,
    "locked_threshold": QC_PARAMS["max_mito_pct"],
    "celltype_col":     CELLTYPE_COL,
    "celltype_source":  "SEA-AD consortium labels" if CELLTYPE_COL else "unavailable",
    "astro_total":      (int(astro_mask.sum()) or None) if CELLTYPE_COL else None,
    "micro_total":      (int(micro_mask.sum()) or None) if CELLTYPE_COL else None,
    "snapshot":         "post count/gene filter, pre mito filter",
    "thresholds":       sweep,
    "niche_check":      "executed at per-study QC (SEA-AD has cell-type labels; Li did not)",
}
print("\n" + json.dumps(qc_trace["mito_sensitivity"], indent=2))

## 6 — Audits: A (niche-critical genes, SEA-AD row) + B (APOE recovered → flip)

### 6a — Write Audit A (SEA-AD row), flip Audit B, append SEA-AD QC trace

**Audit A (partial):** are the 8 niche-critical genes present in SEA-AD's vocab at all? (Full FM-vocab intersection is deferred to the FM-env notebook.) **Audit B:** SEA-AD's donor-level APOE genotype was recovered in 4b → flip its row from `skipped` to `pass`. Li 2025's row is promoted **only if currently `skipped`** (the stale colab_00 pre-check that predates colab_01's recovery) — so a genuine `fail`/`warn` can never be clobbered. `n_donors_with_apoe` is counted from the parsed carrier label (not `.notna()`, which would count `"nan"` strings). Top-level Audit B stays `partial` until Haney (colab_03) lands. All three writes go to the single cumulative `outputs/audit_report.json`.

In [ ]:
from src.data_loaders import NICHE_CRITICAL_GENES

# --- Audit A: niche-critical gene presence (SEA-AD row) -------------------------
if looks_ensembl and SYMBOL_COL and SYMBOL_COL in adata.var.columns:
    gene_pool = set(adata.var[SYMBOL_COL].astype(str))
else:
    gene_pool = set(adata.var_names.astype(str))

per_gene = {g: (g in gene_pool) for g in NICHE_CRITICAL_GENES}
missing  = [g for g, present in per_gene.items() if not present]
audit_a_seaad = {
    "study":           "SEA-AD",
    "vocab_format":    "Ensembl" if looks_ensembl else "HGNC",
    "symbol_col_used": SYMBOL_COL if looks_ensembl else None,
    "n_genes_post_qc": adata.n_vars,
    "per_gene":        per_gene,
    "missing":         missing,
    "status":          "pass" if not missing else "fail",
    "note":            "partial -- FM-vocab intersection deferred to FM env notebook",
}

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

# Schema normalization: n_cells_final must reflect ALL filters, including doublets,
# uniformly across studies so the study-balance audit can trust one field. colab_01
# wrote Li's n_cells_final BEFORE Scrublet (true final lives in n_cells_after_scrublet);
# reconcile every *_qc_trace here so that mismatch can't propagate.
for _k, _v in report.items():
    if (_k.endswith("qc_trace") and isinstance(_v, dict)
            and _v.get("n_cells_after_scrublet") is not None):
        _v["n_cells_final"] = _v["n_cells_after_scrublet"]

audit_a = report.get("audit_a_vocab_niche_critical", {"per_study": {}})
audit_a["per_study"]["SEA-AD"] = audit_a_seaad
audit_a["status"] = "partial"
report["audit_a_vocab_niche_critical"] = audit_a

# --- Audit B: flip APOE-metadata rows that are now recovered --------------------
donor_tab = adata.obs[["donor_id","apoe_genotype","apoe_carrier","diagnosis"]].drop_duplicates("donor_id")
# Count donors with a REAL genotype via the parsed carrier label. apoe_genotype was
# set with .astype(str), so missing values are the literal string "nan" -- which
# .notna() would count as present, overstating coverage. "unknown" == missing.
n_apoe = int((donor_tab["apoe_carrier"] != "unknown").sum())
audit_b = report["audit_b_apoe_metadata"]
for entry in audit_b["per_study"]:
    if entry["study"] == "SEA-AD":
        entry["status"] = "pass"
        entry["n_donors_with_apoe"] = n_apoe
        entry["carrier_breakdown"]  = donor_tab["apoe_carrier"].value_counts().to_dict()
        entry["note"] = "APOE genotype recovered from obs in colab_02 4b"
    elif entry["study"] == "Li 2025" and entry["status"] == "skipped":
        # colab_01 4b genuinely recovered Li's APOE (56/56 donors); the on-disk
        # "skipped" is colab_00's stale generic pre-check. Promote ONLY from
        # "skipped" so a real "fail"/"warn" can never be clobbered by this notebook.
        entry["status"] = "pass"
        entry["note"] = "APOE genotype recovered from GEO series matrix in colab_01 4b (retroactive flip)"
# top-level: pass only when every study passes; Haney still skipped -> partial
statuses = [e["status"] for e in audit_b["per_study"]]
audit_b["status"] = "pass" if all(s == "pass" for s in statuses) else "partial"

report["seaad_qc_trace"] = qc_trace
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_a_seaad, indent=2))
print("\nAudit B per-study statuses:", {e["study"]: e["status"] for e in audit_b["per_study"]})
if missing:
    raise RuntimeError(f"Audit A FAIL -- niche-critical genes missing from SEA-AD vocab: {missing}")

## 7 — Save processed AnnData and clean up raw

### 7a — Save SEA-AD processed AnnData

Sparse CSR enforced before write (memory discipline — the concatenated multi-study object downstream is tight). **ITS reminder:** this keeps *all* cell types (integrate-then-subset is locked; the astro+microglia subset happens after integration, not here). One file `seaad_processed.h5ad`; `region` lives on obs.

In [ ]:
from scipy.sparse import csr_matrix, issparse

if not issparse(adata.X) or adata.X.getformat() != "csr":
    adata.X = csr_matrix(adata.X)

PROCESSED_DIR  = os.path.join(DRIVE_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
PROCESSED_PATH = os.path.join(PROCESSED_DIR, "seaad_processed.h5ad")
adata.write_h5ad(PROCESSED_PATH, compression="gzip")

size_mb = os.path.getsize(PROCESSED_PATH) / 1024**2
print(f"Wrote {PROCESSED_PATH}  ({size_mb:.1f} MB)  shape={adata.shape}")

### 7b — Delete raw download

Per Audit F: raw downloads are `keep=False`, deleted after the processed AnnData is saved. Reclaims the ~6 GB/region SEA-AD footprint from runtime-local disk.

In [ ]:
import shutil

raw_size_gb = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, fs in os.walk(RAW_DIR) for f in fs) / 1024**3
print(f"Raw dir size before delete: {raw_size_gb:.2f} GB")
shutil.rmtree(RAW_DIR)
print(f"Deleted {RAW_DIR}")

## 8 — Handoff to colab_03

### 8a — Summary + commit instructions

Lists artifacts and the explicit `git add / commit / push` commands. Processed h5ad is gitignored (Drive-only); freeze, env JSON, and `audit_report.json` are committed. **Fix vs `colab_01`:** the audit-summary print now summarizes dict entries that have no top-level `status` key (e.g. the QC traces) instead of printing `None`.

In [ ]:
import shlex

with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

def _summ(v):
    if isinstance(v, dict):
        if "status" in v:
            return v["status"]
        if "n_cells_final" in v:                      # a QC trace
            return f"qc_trace (n_cells_final={v['n_cells_final']})"
        return "(dict, no status)"
    return v

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {_summ(v)}")

print("\n=== Artifacts ===")
print(f"  committable:  {FREEZE_PATH}")
print(f"  committable:  {ENV_JSON_PATH}")
print(f"  committable:  {AUDIT_REPORT_PATH}")
print(f"  drive-only:   {PROCESSED_PATH}")

rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_02: SEA-AD QC + Audit A (SEA-AD row) + Audit B flip ({TODAY})"
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
for c in [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {shlex.quote(msg)}",
    f"cd {REPO_PATH} && git push",
]:
    print(f"  {c}")

print("\nNext: colab_03_haney_qc.ipynb (Haney 2024, GEO GSE254205 -- 700 MB h5ad, "
      "APOE in obs; thinnest study at 5 donors/condition -- watch the donor-count audit).")